# Member 4 — Comparison, reproducibility and Phase 1 report
استلم بيانات Member 1، نتائج CNN من Member 2، ونتائج Transfer Learning من Member 3.
هذا الدفتر لا يدرب نموذجًا. يتحقق من البيانات والنتائج، يقارنها، يجمع التوثيق ويجهز مستودعًا نظيفًا.
استورد حزم النتائج ثم ضع الدفاتر الأربعة داخل notebooks/ في مجلد العمل. لا تدّعِ اكتمال التدقيق أو التشغيل دون دليل.

## 1. البيئة واستيراد النتائج

In [ ]:
%pip -q install numpy pandas scipy scikit-learn matplotlib Pillow opencv-python-headless 'transformers==4.49.0'


In [ ]:
from pathlib import Path
import os,json,hashlib,sys,subprocess,platform,random
import numpy as np,pandas as pd
from IPython.display import display,Markdown
ROOT=Path.cwd() if Path.cwd().name=='CameraTrap_Phase1_Team' else Path.cwd()/'CameraTrap_Phase1_Team'
ROOT.mkdir(exist_ok=True);os.chdir(ROOT)
for name in ['handoff','runs','review','submission','docs']:Path(name).mkdir(exist_ok=True)
def read_json(p):return json.loads(Path(p).read_text())
def write_json(p,v):
    Path(p).parent.mkdir(parents=True,exist_ok=True)
    Path(p).write_text(json.dumps(v,indent=2,ensure_ascii=False,allow_nan=False))
def digest(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
    return h.hexdigest()
def environment(member):
    Path(f'handoff/requirements_member{member}.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
    write_json(f'handoff/runtime_member{member}.json',{'python':sys.version,'platform':platform.platform()})
print('Workspace:',ROOT)


In [ ]:

ARCHIVES = []  
import zipfile
for path in ARCHIVES:
    with zipfile.ZipFile(path) as archive:
        
        for item in archive.infolist():
            target=(ROOT/item.filename).resolve()
            assert target.is_relative_to(ROOT.resolve()), 'Unsafe archive path'
            if target.is_file() and not item.is_dir():
                assert hashlib.sha256(archive.read(item)).hexdigest()==digest(target), f'Conflict: {item.filename}'
        archive.extractall(ROOT)
print('Imported:',len(ARCHIVES),'archives')


In [ ]:
release=read_json('handoff/dataset_release.json')
assert release['schema']==2 and release['audit_complete'], 'استخدم إصدار Member 1 الجديد بعد اكتمال التدقيق'
for p,h in release['files'].items():assert Path(p).is_file() and digest(p)==h, f'Changed/missing: {p}'
DATA=Path(release['data_dir']);rows=read_json(DATA/'manifest.json')
assert rows and all(Path(r['path']).exists() for r in rows)
DATA_HASH=digest(DATA/'manifest.json')
CFG=read_json('handoff/data_config.json')
assert CFG['preprocessing_sha256']=='87e3366dd7946e566c3a9238095748eb698ed0b57e2a9a5699d49817c440a5eb'
print('Frames:',len(rows),'| split sizes:',pd.Series([r['split'] for r in rows]).value_counts().to_dict())


In [ ]:
environment(4)

## 2. فحص مستقل ومقارنة التجارب الست
نعيد حساب Accuracy وF1 وECE من التوقعات ونفحص أن إعدادات التدريب والتقسيم مشتركة.

In [ ]:
from collections import defaultdict
from sklearn.metrics import accuracy_score,f1_score
checks=[]
def check(name,passed,details=''):
    checks.append({'check':name,'passed':bool(passed),'details':str(details)})
for key in ['id','camera','event','pixel_sha256']:
    groups=defaultdict(set)
    for r in rows:groups[r[key]].add(r['split'])
    bad=[k for k,v in groups.items() if len(v)>1]
    check(key+'_disjoint',not bad,len(bad))
check('unique_ids',len(rows)==len({r['id'] for r in rows}))
changed=[]
for r in rows:
    if hashlib.md5(Path(r['path']).read_bytes()).hexdigest()!=r['processed_file_md5']:changed.append(r['id'])
check('image_integrity',not changed,len(changed))
audit=read_json(DATA/'audit_report.json')
check('human_audit',audit['status']=='complete' and audit['unresolved_flagged_events']==0)
for split in ['train','val_select','val_calibrate','test']:
    check('empty_animal_support_'+split,{r['animal'] for r in rows if r['split']==split}=={0,1})
records=[];contexts=[]
for member in [2,3]:
    rel=read_json(f'handoff/member{member}_release.json')
    assert rel['schema']==2 and rel['dataset_hash']==DATA_HASH
    for p,h in rel['files'].items():assert Path(p).is_file() and digest(p)==h, f'Changed run artifact: {p}'
    out=Path(rel['output_dir'])
    for task in ['triage','species']:
        for regime in (['scratch'] if member==2 else ['frozen','full']):
            folder=out/task/regime;s=read_json(folder/'summary.json');m=read_json(folder/'test_metrics.json')
            z=np.load(folder/'test_probabilities.npz');y=z['y'];p=z['probabilities'];pred=p.argmax(1)
            confidence=p.max(1);correct=pred==y;bins=np.minimum((confidence*15).astype(int),14)
            ece_value=sum(np.mean(bins==i)*abs(correct[bins==i].mean()-confidence[bins==i].mean()) for i in range(15) if np.any(bins==i))
            f1=f1_score(y,pred,labels=list(range(p.shape[1])),average='macro',zero_division=0)
            check(f'metrics_{task}_{regime}',np.isclose(m['accuracy'],accuracy_score(y,pred)) and
                  np.isclose(m['macro_f1_all_training_classes'],f1) and np.isclose(m['ece'],ece_value))
            check(f'provenance_{task}_{regime}',m['context']==s['context'] and
                  m['signature']=={'data':DATA_HASH,'checkpoint':digest(folder/'best.pt'),'policy':digest(folder/'policy.json')})
            contexts.append({k:v for k,v in s['context'].items() if k!='task'})
            records.append({'member':member,'task':task,'regime':regime,'folder':str(folder),
              'val_macro_f1':s['selection_score'],'test_accuracy':m['accuracy'],
              'test_macro_f1':m['macro_f1_all_training_classes'],'top_k':m['top_k'],
              'top_k_accuracy':m['top_k_accuracy'],'ece':m['ece'],'raw_ece':m['uncalibrated_ece'],
              'coverage':m['coverage'],'unknown_crops':m['unknown_count'],'input_size':m['image_size'],
              'threshold':m['policy'].get('threshold','argmax'),
              'animal_recall':m.get('operating_point',{}).get('animal_recall'),
              'empty_removal':m.get('operating_point',{}).get('empty_removal_rate'),
              'accuracy_ci_camera':str(m['camera_bootstrap'].get('accuracy95')),
              'train_run_id':s['wandb_run_id'],'train_url':s.get('wandb_url') or '',
              'test_run_id':m['wandb_test_run_id'],'test_url':m.get('wandb_test_url') or ''})
check('common_training_configuration',all(c==contexts[0] for c in contexts))
comparison=pd.DataFrame(records)
comparison['selected_on_validation']=False
for task in ['triage','species']:
    idx=comparison.loc[comparison.task==task,'val_macro_f1'].idxmax()
    comparison.loc[idx,'selected_on_validation']=True

acceptance=[]
for row in comparison[comparison.selected_on_validation].to_dict('records'):
    targets=([('animal_recall','animal_recall_min','min'),('empty_removal','empty_removal_min','min')]
             if row['task']=='triage' else [('test_macro_f1','species_macro_f1_min','min'),('ece','species_ece_max','max')])
    for metric,key,direction in targets:
        value=row[metric];target=CFG['acceptance'][key]
        acceptance.append({'task':row['task'],'regime':row['regime'],'metric':metric,'value':value,'target':target,
                           'direction':direction,'met':bool(value>=target if direction=='min' else value<=target)})
pd.DataFrame(acceptance).to_csv('submission/acceptance.csv',index=False)
display(pd.DataFrame(acceptance))
comparison.to_csv('submission/comparison.csv',index=False)
pd.DataFrame(checks).to_csv('review/automatic_checks.csv',index=False)
display(pd.DataFrame(checks));display(comparison)
print('اختيار الأفضل مبني على val_macro_f1 فقط، وليس test.')


## 3. عرض المقارنة وأدلة الأخطاء

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
for task in ['triage','species']:
    table=comparison[comparison.task==task]
    table.plot.bar(x='regime',y=['test_accuracy','test_macro_f1','ece'],title=f'{task}: held-out test, {CFG["image_size"]}px, argmax')
    plt.tight_layout();plt.savefig('submission/'+task+'_comparison.png');plt.show()
for row in comparison[comparison.selected_on_validation].to_dict('records'):
    folder=Path(row['folder']);m=read_json(folder/'test_metrics.json')
    print(row['task'],row['regime'],'| Split:',m['split'],'Input:',m['image_size'],'Policy:',m['policy'])
    display(pd.DataFrame(m['per_class']).T)
    if (folder/'private_errors.jpg').exists():display(Image.open(folder/'private_errors.jpg'))


## 4. عينة مراجعة مستقلة وسجل المشاكل
راجع صور train العشوائية التالية وسجّل قرارك في CSV، ثم أعد قراءة الملف عند إنهاء المراجعة. لا تستخدم هذا لتعديل Test.

In [ ]:
audit_path=Path('review/independent_audit.csv')
if not audit_path.exists():
    events={}
    for r in rows:
        if r['split']=='train':events.setdefault(r['event'],[]).append(r)
    rng=random.Random(2026)
    sample=[rng.choice(events[e]) for e in rng.sample(sorted(events),min(20,len(events)))]
    pd.DataFrame([{'id':r['id'],'label_correct':'','boxes_correct':'','person_visible':'','reviewer':'','notes':''} for r in sample]).to_csv(audit_path,index=False)
by_id={r['id']:r for r in rows}
from PIL import ImageDraw
for ident in pd.read_csv(audit_path).id:
    r=by_id[ident];im=Image.open(r['path']).convert('RGB');draw=ImageDraw.Draw(im)
    for b in r['boxes']:
        x1,y1,x2,y2=b['xyxy'];draw.rectangle((x1*im.width,y1*im.height,x2*im.width,y2*im.height),outline='lime',width=3)
    im.thumbnail((650,420));print(ident,r['labels'] or ['empty']);display(im)
issues=Path('review/issues.csv')
if not issues.exists():pd.DataFrame(columns=['issue_id','owner','problem','status','resolution','evidence']).to_csv(issues,index=False)
print('CSV decisions: label_correct=yes/no, boxes_correct=yes/no/not applicable, person_visible=yes/no.')


## 5. فحص إعادة تحميل الأوزان بعملية Python جديدة
الكود المرسل للعملية ظاهر أدناه ويستخدم نفس Preprocessor. هذه تجربة تحميل وforward فقط؛ لا تثبت إعادة التدريب الكامل. عينة train واحدة لفحص الشكل، وليست دقة species.

In [ ]:
PROBE_SOURCE = 'import cv2, numpy as np, torch\nfrom PIL import Image\nfrom torchvision import transforms\nfrom torch.utils.data import DataLoader\nclass Preprocessor:\n    ""\n    def __init__(self,size):\n        self.size=size\n        self.mean=np.array([.485,.456,.406],np.float32)\n        self.std=np.array([.229,.224,.225],np.float32)\n    def prepare(self,bgr):\n        rgb=cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB);h,w=rgb.shape[:2]\n        interp=cv2.INTER_AREA if h>self.size or w>self.size else cv2.INTER_LINEAR\n        return cv2.resize(rgb,(self.size,self.size),interpolation=interp)\n    def finalize(self,rgb):return (rgb.astype(np.float32)/255.-self.mean)/self.std\n\ndef load_rgb_sample(row):\n    im=cv2.imread(row[\'path\'])\n    if im is None:raise ValueError(\'Unreadable image: \'+row[\'path\'])\n    if row.get(\'crop\') is not None:\n        x1,y1,x2,y2=row[\'crop\'];h,w=im.shape[:2];dx=(x2-x1)*.05;dy=(y2-y1)*.05\n        left,top=max(0,int((x1-dx)*w)),max(0,int((y1-dy)*h))\n        right,bottom=min(w,max(left+1,int((x2+dx)*w))),min(h,max(top+1,int((y2+dy)*h)))\n        im=im[top:bottom,left:right].copy()\n    return im\n\nclass FrameDataset(torch.utils.data.Dataset):\n    ""\n    def __init__(self,samples,size,train=False):\n        self.samples=samples;self.pre=Preprocessor(size)\n        self.tf=transforms.Compose([transforms.RandomHorizontalFlip(),\n                  transforms.ColorJitter(brightness=.15,contrast=.15)]) if train else None\n    def __len__(self):return len(self.samples)\n    def __getitem__(self,i):\n        row=self.samples[i];img=self.pre.prepare(load_rgb_sample(row))\n        if self.tf:img=np.array(self.tf(Image.fromarray(img)))\n        x=torch.from_numpy(np.ascontiguousarray(self.pre.finalize(img))).permute(2,0,1).float()\n        return x,int(row[\'label\']),i\nfrom torch import nn\n\nclass ConvNN(nn.Module):\n    ""\n    def __init__(self,num_classes,size=224,num_filters=16,num_neurons=100):\n        super().__init__()\n        def block(a,b):return nn.Sequential(nn.Conv2d(a,b,3,padding=1),nn.BatchNorm2d(b),nn.ReLU())\n        self.block_1=block(3,num_filters)\n        self.block_2=block(num_filters,2*num_filters)\n        self.block_3=block(2*num_filters,4*num_filters)\n        self.maxpool=nn.MaxPool2d(kernel_size=3,stride=2)\n        self.dropout=nn.Dropout(.2);self.relu=nn.ReLU();self.flat=nn.Flatten()\n        h=size\n        for _ in range(3):h=(h-3)//2+1\n        self.fc1=nn.Linear(4*num_filters*h*h,num_neurons)\n        self.fc2=nn.Linear(num_neurons,num_neurons)\n        self.output=nn.Linear(num_neurons,num_classes)\n    def forward(self,x):\n        x=self.maxpool(self.block_1(x));x=self.maxpool(self.block_2(x));x=self.maxpool(self.block_3(x))\n        x=self.flat(self.dropout(x));x=self.relu(self.fc1(x));x=self.relu(self.fc2(x))\n        return self.output(x)\n\nclass TransferNet(nn.Module):\n    ""\n    def __init__(self,classes,pretrained=True):\n        super().__init__()\n        from transformers import AutoModelForImageClassification,ResNetConfig,ResNetForImageClassification\n        self.net=(AutoModelForImageClassification.from_pretrained(\'microsoft/resnet-50\') if pretrained\n                  else ResNetForImageClassification(ResNetConfig()))\n        self.net.classifier=nn.Sequential(nn.Flatten(start_dim=1),nn.Linear(self.net.config.hidden_sizes[-1],classes))\n        self.net.config.num_labels=classes\n    def forward(self,x):return self.net(x).logits\n    @property\n    def classifier(self):return self.net.classifier\n\ndef make_model(regime,classes,pretrained=True,size=224):\n    return ConvNN(classes,size) if regime==\'scratch\' else TransferNet(classes,pretrained)\n\ndef freeze_backbone(model,freeze):\n    for p in model.net.base_model.parameters():p.requires_grad=not freeze\n    for p in model.classifier.parameters():p.requires_grad=True\nif __name__==\'__main__\':\n    import sys,json\n    checkpoint,image=sys.argv[1:3]\n    torch.set_num_threads(2)\n    ckpt=torch.load(checkpoint,map_location=\'cpu\',weights_only=False)\n    model=make_model(ckpt[\'regime\'],len(ckpt[\'classes\']),pretrained=False,size=ckpt[\'image_size\']).eval()\n    model.load_state_dict(ckpt[\'state_dict\'])\n    x,_,_=FrameDataset([{\'path\':image,\'label\':0,\'crop\':None}],ckpt[\'image_size\'])[0]\n    with torch.no_grad():y=model(x.unsqueeze(0))\n    assert y.shape==(1,len(ckpt[\'classes\'])) and torch.isfinite(y).all()\n    print(json.dumps({\'shape\':list(y.shape),\'finite\':True}))\n'
fresh=[]
sample=next(r['path'] for r in rows if r['split']=='train')
for row in comparison.to_dict('records'):
    checkpoint=Path(row['folder'])/'best.pt'
    try:
        proc=subprocess.run([sys.executable,'-c',PROBE_SOURCE,str(checkpoint),sample],capture_output=True,text=True,timeout=180)
        fresh.append({'checkpoint':str(checkpoint),'sha256':digest(checkpoint),'passed':proc.returncode==0,
                      'stdout':proc.stdout[-1000:],'stderr':proc.stderr[-2000:]})
    except subprocess.TimeoutExpired:
        fresh.append({'checkpoint':str(checkpoint),'sha256':digest(checkpoint),'passed':False,'stderr':'timeout'})
write_json('review/fresh_process_results.json',fresh);display(pd.DataFrame(fresh)[['checkpoint','passed']])


## 6. روابط W&B ومساهمات الفريق
أضف روابط تدريب واختبار التجارب بعد مزامنة offline. السجلات المحلية وحدها لا تمنح رابط dashboard. سجّل إعادة التشغيل في جلسة Colab نظيفة بدقة، ولا تعتبر فحص forward إعادة تدريب.

In [ ]:
links_path=Path('submission/wandb_links.csv')
if not links_path.exists():comparison[['task','regime','train_run_id','train_url','test_run_id','test_url']].to_csv(links_path,index=False)
TEAM={'member1':'','member2':'','member3':'','member4':''}
NOTES={'motivation':'','data_findings':'','model_comparison':'','error_analysis':'','data_improvement_plan':'','limitations':'','conclusion':''}
REPLAY={'reviewer':'','date':'','notebook':'','cells_executed':'','evidence':''}
REVIEWER=''
VISUAL_REVIEW_COMPLETE=False
REPOSITORY_URL='https://github.com/salehdev3/Capstone_5'
write_json('submission/report_inputs.json',{'team':TEAM,'notes':NOTES,'replay':REPLAY,'repository':REPOSITORY_URL})


## 7. قائمة اكتمال المتطلبات
READY تعني اكتمال الشواهد الآلية والحقول؛ لا تضمن جودة التقرير أو درجة الدكتور.

In [ ]:
gaps=[]
automatic=pd.read_csv('review/automatic_checks.csv')
if not automatic.passed.all():gaps.append('Automatic integrity/metrics checks failed')
review=pd.read_csv('review/independent_audit.csv').fillna('')
valid=review.label_correct.eq('yes') & review.boxes_correct.isin(['yes','not applicable']) & review.person_visible.eq('no') & review.reviewer.str.strip().ne('')
if not valid.all():gaps.append('Independent audit pending or flagged')
issues=pd.read_csv('review/issues.csv').fillna('')
if not issues.empty and not (issues.status.eq('resolved') & issues.resolution.str.strip().ne('') & issues.evidence.str.strip().ne('')).all():gaps.append('Unresolved issues')
if len(fresh)!=6 or not all(r['passed'] for r in fresh):gaps.append('Fresh-process inference')
if not REVIEWER.strip() or not VISUAL_REVIEW_COMPLETE:gaps.append('Named visual review signoff')
for k,v in TEAM.items():
    if not v.strip():gaps.append('Team name: '+k)
for k,v in NOTES.items():
    if not v.strip():gaps.append('Report section: '+k)
if not all(v.strip() for v in REPLAY.values()):gaps.append('Clean notebook replay evidence')
for member in [2,3]:
    p=Path(f'runs/member{member}/ERROR_ANALYSIS.md')
    if not p.exists() or 'TODO' in p.read_text():gaps.append(f'Actual error analysis member {member}')
card=Path(release['dataset_card']).read_text()
if 'Pending:' in card:gaps.append('Dataset Card team observations')
blur=pd.read_csv(DATA/'blur_review.csv').fillna('')
if not (blur.reviewer.str.strip().ne('') & blur.decision.isin(['keep','flag']) & blur.notes.str.strip().ne('')).all():gaps.append('Blur review notes')
links=pd.read_csv(links_path).fillna('')
expected={(t,r) for t in ['triage','species'] for r in ['scratch','frozen','full']}
if len(links)!=6 or set(zip(links.task,links.regime))!=expected:gaps.append('Six W&B experiment rows')
for field in ['train_url','test_url']:
    if not links[field].str.match(r'^https://wandb\.ai/[^/]+/[^/]+/runs/[^/]+').all():gaps.append('W&B '+field)
for i in range(1,5):
    if not Path(f'handoff/requirements_member{i}.txt').exists():gaps.append(f'Environment member {i}')
NAMES=['Phase1_Member_1_Data.ipynb', 'Phase1_Member_2_CNN.ipynb', 'Phase1_Member_3_TransferLearning.ipynb', 'Phase1_Member_4_Report.ipynb']
for name in NAMES:
    if not (Path('notebooks')/name).exists():gaps.append('Notebook '+name)
if not REPOSITORY_URL.startswith('https://github.com/'):gaps.append('GitHub repository URL')
STATUS='READY_FOR_TEAM_REVIEW' if not gaps else 'INCOMPLETE'
print(STATUS)
for gap in gaps:print('-',gap)
write_json('handoff/qa_release.json',{'schema':2,'dataset_hash':DATA_HASH,'status':STATUS,'gaps':gaps,
 'reviewer':REVIEWER,'input_hashes':{f'member{i}':digest(f'handoff/member{i}_release.json') for i in [2,3]},
 'scope':'data/metrics checks and fresh-process inference; full replay scope documented separately'})


## 8. التقرير من نتائج التشغيل الفعلية

In [ ]:
def table(df):
    df=df.fillna('');clean=lambda v:str(v).replace('|','/').replace('\n',' ')
    return '\n'.join(['| '+' | '.join(df.columns)+' |','| '+' | '.join(['---']*len(df.columns))+' |']+
                     ['| '+' | '.join(clean(v) for v in row)+' |' for row in df.itertuples(index=False,name=None)])
sections=['# Project 5 — Phase 1 report','Status: '+STATUS,
 'Scope: M1–M3 only. Frame empty/animal classification and species human-box crop classification.',
 '## Contributions',str(TEAM),'## Motivation',NOTES['motivation'] or '[pending]',
 '## Dataset Card',Path(release['dataset_card']).read_text(),NOTES['data_findings'] or '[pending]',
 '## Acceptance criteria',table(pd.DataFrame(acceptance)),
 '## Model comparison',table(comparison.drop(columns=['folder','train_url','test_url'])),
 'Selections above use validation macro-F1 only. Test is held out. Per-class reports and confidence intervals are included in results/.',
 NOTES['model_comparison'] or '[pending]','## Error analysis',NOTES['error_analysis'] or '[pending]']
for member in [2,3]:
    p=Path(f'runs/member{member}/ERROR_ANALYSIS.md')
    sections.append(p.read_text() if p.exists() else '[missing analysis]')
sections+=['## Data improvement plan',NOTES['data_improvement_plan'] or '[pending]',
 '## Limitations',NOTES['limitations'] or '[pending]',
 'The species task uses human-box crops. This does not evaluate a detector-classifier cascade. Camera-based confidence intervals use few sites and are descriptive.',
 '## Reproducibility',str(REPLAY),'Repository: '+(REPOSITORY_URL or '[pending]'),
 '## W&B',table(links),'## Conclusion',NOTES['conclusion'] or '[pending]',
 '## Outstanding evidence','\n'.join('- '+g for g in gaps) or 'No automated completeness gaps.']
Path('submission/REPORT_DRAFT.md').write_text('\n\n'.join(sections))
display(Markdown(Path('submission/REPORT_DRAFT.md').read_text()))


## 9. تجهيز المستودع دون الصور أو الأوزان
تُنسخ الملفات المحددة فقط. دفاتر المستودع تُحذف منها المخرجات التي قد تحتوي صورًا؛ احتفظ بنسخ التنفيذ الخاصة. ارفع هذا المجلد إلى GitHub ثم سجل الرابط وأعد إنشاء التقرير.

In [ ]:
import shutil
repo=Path('submission/repository');repo.mkdir(exist_ok=True)
def collect(source,destination):
    source=Path(source)
    if source.exists():
        target=repo/destination;target.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(source,target)
for name in NAMES:
    p=Path('notebooks')/name
    if p.exists():
        n=read_json(p)
        for cell in n['cells']:
            if cell['cell_type']=='code':cell['outputs']=[];cell['execution_count']=None
        write_json(repo/'notebooks'/name,n)
for p in Path('handoff').glob('requirements_member*.txt'):collect(p,Path('environments')/p.name)
for p in Path('handoff').glob('runtime_member*.json'):collect(p,Path('environments')/p.name)
collect('handoff/data_config.json','config.json')
collect(release['dataset_card'],'DATASET_CARD.md')
for name in ['comparison.csv','acceptance.csv','REPORT_DRAFT.md','wandb_links.csv','report_inputs.json','triage_comparison.png','species_comparison.png']:
    collect(Path('submission')/name,name)
for name in ['automatic_checks.csv','independent_audit.csv','issues.csv','fresh_process_results.json']:
    collect(Path('review')/name,Path('review')/name)
for name in ['class_balance_proportions.csv','species_crop_support.csv','audit_report.json','blur_review.csv']:
    collect(DATA/name,Path('data_evidence')/name)
weights=[]
for row in comparison.to_dict('records'):
    folder=Path(row['folder']);destination=Path('results')/row['task']/row['regime']
    for name in ['summary.json','test_metrics.json','history.csv','policy.json','confusion.png','learning_loss.png','learning_accuracy.png','reliability.png','validation_threshold_sweep.csv']:
        collect(folder/name,destination/name)
    weights.append({'path':str(folder/'best.pt'),'sha256':digest(folder/'best.pt'),'access':'private team storage'})
write_json(repo/'weights_manifest.json',weights)
(repo/'.gitignore').write_text('data/\nruns/\nwandb/\n*.pt\n*.pth\n*.jpg\n*.jpeg\n.env\n__pycache__/\n.ipynb_checkpoints/\n')
(repo/'README.md').write_text("# Project 5 — Phase 1\n\nRun notebooks 1 → (2 and 3) → 4. Use the same Member1 dataset release for both training notebooks. Upload private result archives to Colab and set ARCHIVES in the import cell; no Drive mount or external code modules are required. Internet and Colab's torch/torchvision are required; use GPU for training.\n\nMember1 uses CCT20 from https://lila.science/datasets/caltech-camera-traps. Follow dataset licence and course non-redistribution policy. Do not commit images or raw archives. The Dataset Card records filtering and limitations.\n\nAll three notebooks embed identical OpenCV preprocessing. Split is stratified by camera groups; bursts stay together. One shared config defines all six experiments. Validation selects models; calibration chooses temperature/triage threshold; test is evaluated once.\n\nSee environments/ for the actual resolved package versions and Python/platform of each member. Install one role's environment in a compatible runtime. No single cross-platform lock is claimed. The initial install cells describe required packages; actual resolved pins are captured after installation.\n\nReport and metrics are from actual execution, with pending sections visible. Test accuracy intervals include camera bootstrap and Wilson (the latter assumes independent samples). Human audit, error discussion, clean replay and W&B sync require team action. Fresh-process inference alone is not full training reproduction.\n\nWeights remain private; weights_manifest.json provides hashes and paths. Repository notebooks have outputs stripped; keep executed copies privately for assessment. This repository covers M1–M3 only, without detector training or deployment.\n")
assert not list(repo.rglob('*.pt')) and not list(repo.rglob('*.jpg'))
write_json('handoff/integration_release.json',{'schema':2,'scope':'Phase 1','status':STATUS,'gaps':gaps,
 'dataset_hash':DATA_HASH,'report_sha256':digest(repo/'REPORT_DRAFT.md'),'repository':REPOSITORY_URL})
archive=shutil.make_archive(str(ROOT.parent/'Phase1_Submission'),'zip',root_dir=repo)
print('Prepared:',repo.resolve(),'|',archive)
from google.colab import files
files.download(archive)
